In [ ]:
!pip install tensorflow


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [ ]:
# Paramètres
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 10
SEED = 42
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Charger les données
df = pd.read_csv("train.csv")
df = df.groupby('diagnosis', group_keys=False).apply(lambda x: x.sample(frac=0.1, random_state=SEED))
df['file_path'] = df['id_code'].apply(lambda x: os.path.join("train_images", f"{x}.png"))

In [ ]:
# Encode les labels
le = LabelEncoder()
df['label'] = le.fit_transform(df['diagnosis'])
num_classes = df['label'].nunique()

In [ ]:
# Charger les images
def load_images(df):
    X = []
    for path in df['file_path']:
        img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE))
        img = img_to_array(img)
        img = preprocess_input(img)
        X.append(img)
    return np.array(X)

In [ ]:
print("Loading images...")
X = load_images(df)
y = to_categorical(df['label'], num_classes=num_classes)

# Split train/validation
X_train, X_val, y_train, y_val = train_test_split(X, y, stratify=y, test_size=0.15, random_state=SEED)

In [ ]:
# Fonction pour créer un modèle ResNet50
def create_resnet50_model(name="resnet50"):
    base_model = ResNet50(include_top=False, weights='imagenet', input_shape=(IMG_SIZE, IMG_SIZE, 3))
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    output = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=base_model.input, outputs=output, name=name)
    for layer in base_model.layers:
        layer.trainable = False
    model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# Créer les 3 modèles
models = [create_resnet50_model(f"resnet50_{i}") for i in range(3)]

In [ ]:
# Callbacks
callbacks = [
    ReduceLROnPlateau(patience=2, factor=0.5),
    EarlyStopping(patience=4, restore_best_weights=True)
]

In [ ]:
# Entraîner chaque modèle
histories = []
for i, model in enumerate(models):
    print(f"\nTraining model {i+1}")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1
    )
    histories.append(history)

In [ ]:
# Évaluer les modèles
for i, model in enumerate(models):
    loss, acc = model.evaluate(X_val, y_val, verbose=0)
    print(f"Model {i+1} accuracy: {acc:.4f}")


In [ ]:
# Option : moyenne des prédictions
print("\nAverage Ensemble Accuracy:")
preds = [model.predict(X_val) for model in models]
avg_preds = np.mean(preds, axis=0)
final_acc = np.mean(np.argmax(avg_preds, axis=1) == np.argmax(y_val, axis=1))
print(f"Ensembled Accuracy: {final_acc:.4f}")

In [ ]:
# Visualisation des courbes d'entraînement et de validation
for i, history in enumerate(histories):
    plt.figure(figsize=(12, 4))
    plt.suptitle(f'Model {i+1} - Performance')

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.title('Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Labels vrais et prédits (depuis les prédictions moyennes)
y_true = np.argmax(y_val, axis=1)
y_pred = np.argmax(avg_preds, axis=1)

# Matrice de confusion
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)

# Affichage
plt.figure(figsize=(8, 6))
disp.plot(cmap='Blues', values_format='d')
plt.title("Confusion Matrix - Ensemble des 3 modèles")
plt.show()


In [ ]:
from sklearn.metrics import classification_report

for i, model in enumerate(models):
    preds = model.predict(X_val)
    y_pred_model = np.argmax(preds, axis=1)
    print(f"\nClassification Report - Model {i+1}:")
    print(classification_report(y_true, y_pred_model, target_names=le.classes_.astype(str)))


In [ ]:
from sklearn.metrics import classification_report

# Générer le rapport de classification
report = classification_report(y_true, y_pred, target_names=le.classes_.astype(str))

# Afficher
print("Classification Report (Ensemble des 3 modèles):")
print(report)
